# Event-Level ParT Classifier — Evaluation

Evaluate the trained event-level ParticleTransformer binary classifier (HH→4b vs QCD).

Sections:
1. Load model checkpoint from W&B
2. Run inference on validation set
3. ROC curve (signal efficiency vs QCD rejection)
4. Score distributions
5. HT sculpting check
6. Baseline comparison (≥3 b-tagged jets)

In [3]:
import json
import os
import glob
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib

import matplotlib.pyplot as plt
import wandb
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, auc
from torch.utils.data import DataLoader

# Ensure repository root is importable even when notebook cwd is notebooks/
import os, pathlib as _pl

# Navigate to root-obj-perf/ regardless of where the Jupyter kernel starts
_cwd = _pl.Path.cwd()
if _cwd.name == "notebooks" and _cwd.parent.name == "root-obj-perf":
    os.chdir("..")
elif _cwd.name != "root-obj-perf" and (_cwd / "root-obj-perf").exists():
    os.chdir("root-obj-perf")
print(f"Working directory: {os.getcwd()}")

from model.parT import ParticleTransformer
from data_pipeline.datasets import StratifiedJetDataset, precollated_collate
from data_pipeline.splitting import stratified_split

## 1. Load Model

In [4]:
torch.backends.cuda.matmul.allow_tf32 = True

CONFIG_CANDIDATES = [Path("config_event.json"), Path("../config_event.json")]
CONFIG_PATH = next((p for p in CONFIG_CANDIDATES if p.exists()), None)
if CONFIG_PATH is None:
    raise FileNotFoundError(
        "Could not find config_event.json in current or parent directory"
    )

with open(CONFIG_PATH) as f:
    cfg = json.load(f)

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Using config: {CONFIG_PATH}")

# Load checkpoint artifact from W&B
artifact_tag = cfg["wandb"].get("ckpt_type", "best")
artifact_ref = (
    f"{cfg['wandb']['entity']}/{cfg['wandb']['project']}/"
    f"{cfg['wandb']['artifact_name']}:{artifact_tag}"
)
api = wandb.Api()
artifact = api.artifact(artifact_ref, type="model")
artifact_dir = artifact.download()

ckpt_candidates = sorted(
    glob.glob(os.path.join(artifact_dir, "**", "*.pth"), recursive=True)
)
if not ckpt_candidates:
    raise FileNotFoundError(f"No .pth checkpoint found inside artifact: {artifact_ref}")
ckpt_path = ckpt_candidates[0]
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

model = ParticleTransformer(
    input_dim=cfg["input_dim"],
    embed_dim=cfg["model"]["embed_dim"],
    num_pairwise_feat=cfg["model"].get("num_pairwise_feat", 7),
    num_heads=cfg["model"]["num_heads"],
    num_layers=cfg["model"]["num_layers"],
    num_cls_layers=cfg["model"]["num_cls_layers"],
    dropout=cfg["model"]["dropout"],
    num_classes=cfg["model"]["num_classes"],
    use_batch_norm=cfg["model"].get("use_batch_norm", True),
    pt_regression=cfg["model"].get("pt_regression", False),
    quantile_regression=cfg["model"].get("quantile_regression", False),
).to(device)

state_dict = ckpt.get("model_state_dict", ckpt)
model.load_state_dict(state_dict)
model.eval()

epoch = ckpt.get("epoch", None)
val_auc = ckpt.get("val_auc", None)
if epoch is not None and val_auc is not None:
    print(f"Loaded model from epoch {epoch + 1}, val AUC={val_auc:.4f}")
else:
    print(f"Loaded model from checkpoint: {ckpt_path}")

## 2. Inference on Validation Set

In [5]:
from tqdm import tqdm
dataset = StratifiedJetDataset(cfg["training"]["data"]["data_path"])
_, val_ds, _, val_indices, _ = stratified_split(
    dataset, cfg["training"]["data"]["val_split"], num_classes=1
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg["training"]["batch_size"] * 2,
    shuffle=False,
    num_workers=0,
    collate_fn=precollated_collate,
)

all_scores, all_labels, all_qcd_w, all_ht, all_kin_weights = [], [], [], [], []
all_constituents_batches, all_masks_batches = [], []

with torch.no_grad():
    for X_batch, y_batch, mask_batch, kin_weights, jet_pt, _, _, qcd_weights in tqdm(val_loader):
        X_batch = X_batch.to(device)
        mask_batch = mask_batch.to(device)

        outputs = model(X_batch, particle_mask=mask_batch)
        cls_output = outputs["classification"] if isinstance(outputs, dict) else outputs
        batch_scores = torch.sigmoid(cls_output).cpu().numpy()

        all_scores.append(batch_scores)
        all_labels.append(y_batch.numpy())
        all_qcd_w.append(qcd_weights.numpy())
        all_ht.append(jet_pt.numpy())  # jet_pt stores HT
        all_kin_weights.append(kin_weights.numpy())
        all_constituents_batches.append(X_batch.cpu())
        all_masks_batches.append(mask_batch.cpu())

scores = np.concatenate(all_scores).ravel()
labels = np.concatenate(all_labels).ravel()
qcd_w = np.concatenate(all_qcd_w).ravel()
ht = np.concatenate(all_ht).ravel()
kin_weights = np.concatenate(all_kin_weights).ravel()

# Tensors used by interpretability/behavior diagnostics cells.
all_constituents = torch.cat(all_constituents_batches, dim=0)
all_masks = torch.cat(all_masks_batches, dim=0).bool()
all_outputs = scores.copy()
# all_max_etas = np.abs(all_constituents[..., 2][all_masks].max(dim=1).values.numpy())

print(f"Inference done: {len(scores)} events")
print(f"  Signal: {(labels==1).sum()}, Background: {(labels==0).sum()}")
print(f"  Constituents tensor shape: {tuple(all_constituents.shape)}")

In [6]:
# Extract run_id from artifact path for plot directory
run_id = cfg.get("exp_name", "unknown_run").replace("/", "_")
artifact_name = cfg.get("wandb", {}).get("artifact_name", "unknown_artifact")
artifact_type = cfg.get("wandb", {}).get("ckpt_type", "unknown_type")
plot_dir = f"../Updates/plots_{run_id}/{artifact_name}:{artifact_type}"
os.makedirs(plot_dir, exist_ok=True)
print(f"Saving plots to: {plot_dir}")


def save_fig(fig, name):
    """Save figure to plot directory."""
    filepath = os.path.join(plot_dir, f"{name}.png")
    fig.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"  Saved: {name}.png")

## 3. ROC Curve

In [7]:
import pandas as pd
import seaborn as sns

import importlib
import evaluation.roc
importlib.reload(evaluation.roc)
from evaluation.roc import calculate_trained_roc_2d_bins

fpr, tpr, thresholds = roc_curve(labels, scores, sample_weight=qcd_w)
auc_val = roc_auc_score(labels, scores, sample_weight=qcd_w)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))



# Standard ROC
ax1.plot(fpr, tpr, label=f"Event ParT (AUC={auc_val:.4f})")
ax1.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax1.set_xlabel("False Positive Rate (QCD efficiency)")
ax1.set_ylabel("True Positive Rate (Signal efficiency)")
ax1.set_title("ROC Curve")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Signal eff vs QCD rejection
nonzero = fpr > 0
ax2.plot(tpr[nonzero], 1.0 / fpr[nonzero], label=f"Event ParT (AUC={auc_val:.4f})")
ax2.set_xlabel("Signal Efficiency")
ax2.set_ylabel("QCD Rejection (1/FPR)")
ax2.set_yscale("log")
ax2.set_title("Signal Efficiency vs QCD Rejection")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, "event_roc_curve")
plt.show()
print(f"AUC = {auc_val:.4f}")

from plotting.base import plot_roc_comparison
roc_fig = plot_roc_comparison(
    [
        ("Trained ParT", (fpr, tpr, auc_val, thresholds)),
    ],
    working_point=0.1,
    return_fig=True,
)

save_fig(roc_fig, "event_roc_comparison")
roc_fig.show()

pt_ranges = [
    (25, 100),
    (100, 200),
    (200, 300),
    (300, 400),
    (400, 500),
    (500, np.inf),
    (25, np.inf),
]
eta_ranges = [(0, 0.5), (0.5, 1.0), (1.0, 1.5), (1.5, 2.4), (0, 1.5), (0, 2.4)]


# Use reconstructed jet kinematics
all_event_max_eta = np.abs(all_constituents[..., 2].numpy()).max(axis=1)

auc_matrix, count_matrix = calculate_trained_roc_2d_bins(
    pt_ranges,
    eta_ranges,
    ht,
    all_event_max_eta,
    labels,
    scores,
    weights=qcd_w,
)

# Plot heatmap
pt_labels = [f"[{low},{high})" for low, high in pt_ranges]
eta_labels = [f"[{low},{high})" for low, high in eta_ranges]
annot = np.empty_like(auc_matrix, dtype=object)
for i in range(auc_matrix.shape[0]):
    for j in range(auc_matrix.shape[1]):
        if np.isnan(auc_matrix[i, j]):
            annot[i, j] = f"N={int(count_matrix[i, j])}\n(N/A)"
        else:
            annot[i, j] = f"{auc_matrix[i, j]:.3f}\n(N={int(count_matrix[i, j])})"

fig, ax = plt.subplots(figsize=(12, 9))
df = pd.DataFrame(auc_matrix, index=pt_labels, columns=eta_labels)
sns.heatmap(
    df,
    annot=annot,
    fmt="",
    cmap="viridis",
    vmin=0.5,
    vmax=1.0,
    ax=ax,
    cbar_kws={"label": "AUC"},
)
ax.set_xlabel(r"$|\eta|$")
ax.set_ylabel("Jet pT [GeV]")
ax.set_title(r"Trained ParT: AUC in pT vs $|\eta|$ Bins")
plt.tight_layout()
save_fig(fig, "auc_heatmap_pt_eta")
plt.show()


## 4. Score Distributions

In [8]:
fig, ax = plt.subplots(figsize=(8, 5))

bins = np.linspace(0, 1, 51)
ax.hist(
    scores[labels == 1],
    bins=bins,
    alpha=0.6,
    label="HH→4b (signal)",
    density=True,
    histtype="stepfilled",
    color="tab:blue",
)
ax.hist(
    scores[labels == 0],
    bins=bins,
    alpha=0.6,
    label="QCD (background)",
    density=True,
    histtype="stepfilled",
    color="tab:orange",
    weights=qcd_w[labels == 0],
)

ax.set_xlabel("Classifier Score")
ax.set_ylabel("Normalised Density")
ax.set_title("Event-Level ParT Score Distributions")
ax.legend()
# ax.set_yscale("log")
ax.grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, "event_scores")
plt.show()

## 5. HT Sculpting Check

Verify the classifier score is not strongly correlated with event HT for QCD.

In [9]:
bkg_mask = labels == 0

fig, ax = plt.subplots(figsize=(8, 6))
h = ax.hist2d(
    ht[bkg_mask],
    scores[bkg_mask],
    bins=[50, 50],
    range=[[0, np.percentile(ht[bkg_mask], 99)], [0, 1]],
    cmap="viridis",
    weights=qcd_w[bkg_mask],
)
plt.colorbar(h[3], ax=ax, label="Weighted counts")
ax.set_xlabel("Event HT (GeV)")
ax.set_ylabel("Classifier Score")
ax.set_title("QCD: Score vs HT (sculpting check)")

plt.tight_layout()
save_fig(fig, "event_ht_sculpting_qcd")
plt.show()

bkg_mask = labels == 0

fig, ax = plt.subplots(figsize=(8, 6))
h = ax.hist2d(
    ht[labels==1],
    scores[labels==1],
    bins=[50, 50],
    range=[[0, np.percentile(ht[labels==1], 99)], [0, 1]],
    cmap="viridis",
    weights=qcd_w[labels==1],
)
plt.colorbar(h[3], ax=ax, label="Weighted counts")
ax.set_xlabel("Event HT (GeV)")
ax.set_ylabel("Classifier Score")
ax.set_title("Signal: Score vs HT (sculpting check)")

plt.tight_layout()
save_fig(fig, "event_ht_sculpting_signal")
plt.show()

## 6. Baseline: b-tag Cut Comparison

Compare the ParT classifier to a simple "≥3 b-tagged jets" cut.

In [10]:
# The baseline is effectively the trigger requirement itself.
# If we ran with trigger, all events already pass ≥3 b-tags.
# For a meaningful comparison, compute a score based on the number of
# b-tagged jets (from the dataset metadata if available) or just report
# the ParT AUC against the trigger efficiency.

print(f"Event-level ParT AUC: {auc_val:.4f}")
print(f"\nNote: If all events already pass the trigger (≥3 b-tags),")
print(f"the ParT classifier provides discrimination *beyond* the trigger cut.")
print(f"\nPrecision-Recall AUC:")
prec, rec, _ = precision_recall_curve(labels, scores, sample_weight=qcd_w)
pr_auc = auc(rec, prec)
print(f"  PR-AUC = {pr_auc:.4f}")

In [11]:
plt.hist(ht[bkg_mask], bins=50,  range=(0, np.percentile(ht[bkg_mask], 90)), label="QCD (background)", histtype="step", density=True)
plt.hist(ht[labels == 1], bins=50, label="Signal (HH→4b)", range=(0, np.percentile(ht[bkg_mask], 90)), histtype="step", density=True)
plt.hist(ht, bins=50,  range=(0, np.percentile(ht[bkg_mask], 90)), label="All events", histtype="step", density=True)
plt.xlabel("Event HT (GeV)")
plt.ylabel("Normalised density")
plt.title("HT distribution")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.legend()
plt.show()

In [12]:
plt.hist(ht[bkg_mask], bins=50, weights=qcd_w[bkg_mask], range=(0, np.percentile(ht[bkg_mask], 90)), label="QCD (background)", color="tab:orange", histtype="step", density=True)
plt.hist(ht[labels == 1], bins=50, label="Signal (HH→4b)", color="tab:blue", range=(0, np.percentile(ht[bkg_mask], 90)), histtype="step", density=True)
plt.hist(ht, bins=50, weights=qcd_w, label="All events", color="tab:green", range=(0, np.percentile(ht[bkg_mask], 90)), histtype="step", density=True)
plt.xlabel("Event HT (GeV)")
plt.ylabel("Weighted counts")
plt.title("Cross section weighted HT distribution")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.legend()
plt.show()

In [13]:
plt.hist(ht[bkg_mask], bins=50, weights=kin_weights[bkg_mask], range=(0, np.percentile(ht[bkg_mask], 90)), label="QCD (background)", histtype="step", density=True)
plt.hist(ht[labels == 1], bins=50, weights=kin_weights[labels == 1], label="Signal (HH→4b)", range=(0, np.percentile(ht[bkg_mask], 90)), histtype="step", density=True)
plt.hist(ht, bins=50, weights=kin_weights, range=(0, np.percentile(ht[bkg_mask], 90)), label="All events", histtype="step", density=True)
plt.xlabel("Event HT (GeV)")
plt.ylabel("Normalised density")
plt.title("Kinetically reweighted HT distribution (normalised)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.legend()
plt.show()

## 7. Input Importance And Model Behavior

Equivalent analyses to Cells 34-37 in `test_trained_part`, adapted to the event-level classifier notebook.

In [14]:
from evaluation.attention import compute_pairwise_features, forward_with_attention

print("=" * 60)
print("ATTENTION AND PAIRWISE FEATURE ANALYSIS")
print("=" * 60)

# Balanced signal/background subset for qualitative attention inspection.
n_target = 5
signal_indices = np.where(labels == 1)[0][:n_target]
background_indices = np.where(labels == 0)[0][:n_target]
n_sig = len(signal_indices)
n_bkg = len(background_indices)
n_plot = min(n_sig, n_bkg)

if n_plot == 0:
    raise RuntimeError("Need at least one signal and one background event for attention comparison.")

signal_indices = signal_indices[:n_plot]
background_indices = background_indices[:n_plot]
sample_indices = np.concatenate([signal_indices, background_indices])

sample_x = all_constituents[sample_indices].float().to(device)
sample_mask = all_masks[sample_indices].to(device)

pairwise_feats, _ = compute_pairwise_features(sample_x.cpu(), sample_mask.cpu())
model.eval()
with torch.no_grad():
    attention_maps, _ = forward_with_attention(model, sample_x, sample_mask)

print(f"Analyzed {len(sample_indices)} events ({n_plot} signal, {n_plot} background)")
print(f"Particle attention layers: {len(attention_maps['particle_attn'])}")
print(f"Class attention layers: {len(attention_maps['class_attn'])}")

print("\nComputing pairwise feature distributions...")
n_jets_for_dist = min(1000, len(all_constituents))
dist_x = all_constituents[:n_jets_for_dist]
dist_mask = all_masks[:n_jets_for_dist]
dist_labels = labels[:n_jets_for_dist]
pairwise_dist, _ = compute_pairwise_features(dist_x, dist_mask)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

feature_names_pairwise = [
    ("log_delta_R", "log(Delta R)"),
    ("log_k_t", "log(kT)"),
    ("log_z", "log(z)"),
    ("log_m_2", "log(m^2)"),
    ("d_dxy", "Delta d_xy"),
    ("d_z0", "Delta z0"),
    ("q_ij", "q_i * q_j"),
]

for idx, (feat_key, feat_label) in enumerate(feature_names_pairwise):
    ax = axes[idx]
    feat = pairwise_dist[feat_key].numpy()
    mask_np = dist_mask.numpy()

    sig_vals = []
    bkg_vals = []

    for i in range(len(feat)):
        n_valid = int(mask_np[i].sum())
        if n_valid > 1:
            fvals = feat[i, :n_valid, :n_valid]
            upper_tri = fvals[np.triu_indices(n_valid, k=1)]
            upper_tri = upper_tri[np.isfinite(upper_tri)]
            if dist_labels[i] == 1:
                sig_vals.extend(upper_tri)
            else:
                bkg_vals.extend(upper_tri)

    sig_vals = np.asarray(sig_vals)
    bkg_vals = np.asarray(bkg_vals)

    if len(sig_vals) > 0 and len(bkg_vals) > 0:
        range_min = min(np.percentile(sig_vals, 1), np.percentile(bkg_vals, 1))
        range_max = max(np.percentile(sig_vals, 99), np.percentile(bkg_vals, 99))
        ax.hist(sig_vals, bins=50, range=(range_min, range_max), histtype="step", color="tab:blue", density=True, label="Signal")
        ax.hist(bkg_vals, bins=50, range=(range_min, range_max), histtype="step", color="tab:orange", density=True, label="Background")

    ax.set_xlabel(feat_label)
    ax.set_ylabel("Density")
    ax.set_title(feat_label)
    ax.legend(fontsize=8)

axes[-1].axis("off")
plt.tight_layout()
save_fig(fig, "event_pairwise_feature_distributions")
plt.show()

print("\nPlotting class-token attention weights...")
fig, axes = plt.subplots(2, n_plot, figsize=(4 * n_plot, 8), squeeze=False)
for i in range(n_plot):
    sig_idx = i
    bkg_idx = n_plot + i

    cls_attn_sig = attention_maps["class_attn"][-1][sig_idx, :, 0, 1:].mean(dim=0).numpy()
    n_valid_sig = int(sample_mask[sig_idx].sum().item())
    ax = axes[0, i]
    ax.bar(np.arange(n_valid_sig), cls_attn_sig[:n_valid_sig], color="tab:blue")
    ax.set_title(f"Signal {i+1}")
    ax.set_xlabel("Constituent index")
    ax.set_ylabel("Attention")

    cls_attn_bkg = attention_maps["class_attn"][-1][bkg_idx, :, 0, 1:].mean(dim=0).numpy()
    n_valid_bkg = int(sample_mask[bkg_idx].sum().item())
    ax = axes[1, i]
    ax.bar(np.arange(n_valid_bkg), cls_attn_bkg[:n_valid_bkg], color="tab:orange")
    ax.set_title(f"Background {i+1}")
    ax.set_xlabel("Constituent index")
    ax.set_ylabel("Attention")

fig.suptitle("Class-token attention over constituents", fontsize=14)
plt.tight_layout()
save_fig(fig, "event_class_attention_weights")
plt.show()

In [15]:
from evaluation.attention import forward_with_activations, compute_separability

print("=" * 60)
print("LAYER ACTIVATION ANALYSIS")
print("=" * 60)

n_vis = min(500, len(all_constituents))
vis_x = all_constituents[:n_vis].float().to(device)
vis_mask = all_masks[:n_vis].to(device)
vis_labels = labels[:n_vis]

print(f"Analyzing activations for {n_vis} events")
print(f"  Signal: {(vis_labels == 1).sum()}")
print(f"  Background: {(vis_labels == 0).sum()}")

with torch.no_grad():
    activations = forward_with_activations(model, vis_x, vis_mask)

layer_names = (
    ["embedding"]
    + [f"particle_attn_{i+1}" for i in range(len(activations["particle_attn_layers"]))]
    + ["pre_cls"]
    + [f"cls_attn_{i+1}" for i in range(len(activations["cls_attn_layers"]))]
    + ["final_cls_token"]
)

layer_activations = [activations["embedding"]]
layer_activations.extend(activations["particle_attn_layers"])
layer_activations.append(activations["pre_cls"])
layer_activations.extend(activations["cls_attn_layers"])
layer_activations.append(activations["final_cls_token"].unsqueeze(1))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, (name, act) in enumerate(zip(layer_names, layer_activations)):
    if idx >= len(axes):
        break

    ax = axes[idx]
    act_flat = act.mean(dim=1).numpy() if len(act.shape) == 3 else act.numpy()
    sig_act = act_flat[vis_labels == 1].flatten()
    bkg_act = act_flat[vis_labels == 0].flatten()

    if len(sig_act) == 0 or len(bkg_act) == 0:
        ax.set_title(f"{name} (insufficient class stats)")
        ax.axis("off")
        continue

    range_min = min(np.percentile(sig_act, 1), np.percentile(bkg_act, 1))
    range_max = max(np.percentile(sig_act, 99), np.percentile(bkg_act, 99))

    ax.hist(sig_act, bins=50, range=(range_min, range_max), histtype="step", label="Signal", color="tab:blue", density=True)
    ax.hist(bkg_act, bins=50, range=(range_min, range_max), histtype="step", label="Background", color="tab:orange", density=True)
    ax.set_xlabel("Activation")
    ax.set_ylabel("Density")
    ax.set_title(name)
    ax.legend(fontsize=8)

for idx in range(len(layer_names), len(axes)):
    axes[idx].axis("off")

plt.suptitle("Activation magnitude distributions across layers", fontsize=14)
plt.tight_layout()
save_fig(fig, "event_activation_magnitude_distributions")
plt.show()

print("\nComputing layer-wise separability (Fisher ratio)...")
sep_names = (
    ["input_processed", "embedding"]
    + [f"particle_attn_{i+1}" for i in range(len(activations["particle_attn_layers"]))]
    + ["pre_cls"]
    + [f"cls_attn_{i+1}" for i in range(len(activations["cls_attn_layers"]))]
    + ["final_cls_token"]
)
sep_acts = [
    activations["input_processed"].mean(dim=1).numpy(),
    activations["embedding"].mean(dim=1).numpy(),
]
sep_acts.extend([a.mean(dim=1).numpy() for a in activations["particle_attn_layers"]])
sep_acts.append(activations["pre_cls"].mean(dim=1).numpy())
sep_acts.extend([a.mean(dim=1).numpy() for a in activations["cls_attn_layers"]])
sep_acts.append(activations["final_cls_token"].numpy())

mean_fisher_vals = []
max_fisher_vals = []
for arr in sep_acts:
    mean_fisher, max_fisher, _ = compute_separability(arr, vis_labels)
    mean_fisher_vals.append(mean_fisher)
    max_fisher_vals.append(max_fisher)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
x = np.arange(len(sep_names))
ax1.bar(x, mean_fisher_vals, color="tab:blue", alpha=0.8)
ax1.set_title("Mean Fisher separability by layer")
ax1.set_ylabel("Mean Fisher ratio")
ax1.set_xticks(x)
ax1.set_xticklabels([n.replace("_", "\n") for n in sep_names], rotation=45, ha="right", fontsize=8)

ax2.bar(x, max_fisher_vals, color="tab:orange", alpha=0.8)
ax2.set_title("Max-neuron Fisher separability by layer")
ax2.set_ylabel("Max Fisher ratio")
ax2.set_xticks(x)
ax2.set_xticklabels([n.replace("_", "\n") for n in sep_names], rotation=45, ha="right", fontsize=8)

plt.tight_layout()
save_fig(fig, "event_layer_separability")
plt.show()

In [16]:
from scipy.stats import ks_2samp

print("=" * 60)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 60)

base_feature_names = [
    "Mass",
    "pT",
    "eta",
    "phi",
    "d_xy",
    "z_0",
    "Charge",
    "log(pT_rel)",
    "Delta_eta",
    "Delta_phi",
    "PUPPI_weight",
    "log(DeltaR)",
    "is_electron",
    "is_muon",
    "is_neutral_hadron",
    "is_photon",
    "is_charged_hadron",
]

n_features_total = int(all_constituents.shape[2])
if n_features_total <= len(base_feature_names):
    input_feature_names = base_feature_names[:n_features_total]
else:
    input_feature_names = base_feature_names + [
        f"Feature_{i}" for i in range(len(base_feature_names), n_features_total)
    ]

max_features_for_importance = min(n_features_total, 24)
feature_names_used = input_feature_names[:max_features_for_importance]
if max_features_for_importance < n_features_total:
    print(f"Analyzing first {max_features_for_importance}/{n_features_total} features for runtime.")
else:
    print(f"Analyzing {max_features_for_importance} features.")

# 1) Gradient-based importance
print("\n1. Computing gradient-based importance...")
n_grad_samples = min(400, len(all_constituents))
grad_x = all_constituents[:n_grad_samples].clone().float().to(device)
grad_mask = all_masks[:n_grad_samples].to(device)
grad_labels = labels[:n_grad_samples]
grad_x.requires_grad_(True)

model.eval()
out = model(grad_x, particle_mask=grad_mask)
logits = out["classification"] if isinstance(out, dict) else out
probs = torch.sigmoid(logits).squeeze()

model.zero_grad(set_to_none=True)
if grad_x.grad is not None:
    grad_x.grad = None
probs.sum().backward()

grads_abs = grad_x.grad.detach().abs().cpu().numpy()
mask_np = grad_mask.cpu().numpy().astype(bool)

grad_importance = np.zeros(max_features_for_importance)
grad_importance_signal = np.zeros(max_features_for_importance)
grad_importance_background = np.zeros(max_features_for_importance)

for feat_idx in range(max_features_for_importance):
    valid_vals = grads_abs[:, :, feat_idx][mask_np]
    grad_importance[feat_idx] = float(valid_vals.mean()) if valid_vals.size > 0 else 0.0

    sig_vals = []
    bkg_vals = []
    for i in range(n_grad_samples):
        n_valid = int(mask_np[i].sum())
        if n_valid == 0:
            continue
        vals = grads_abs[i, :n_valid, feat_idx]
        if grad_labels[i] == 1:
            sig_vals.extend(vals)
        else:
            bkg_vals.extend(vals)

    grad_importance_signal[feat_idx] = float(np.mean(sig_vals)) if len(sig_vals) > 0 else 0.0
    grad_importance_background[feat_idx] = float(np.mean(bkg_vals)) if len(bkg_vals) > 0 else 0.0

if grad_importance.sum() > 0:
    grad_importance = grad_importance / grad_importance.sum()
if grad_importance_signal.sum() > 0:
    grad_importance_signal = grad_importance_signal / grad_importance_signal.sum()
if grad_importance_background.sum() > 0:
    grad_importance_background = grad_importance_background / grad_importance_background.sum()

# 2) Permutation AUC-drop importance
print("2. Computing permutation importance...")
n_perm_samples = min(800, len(all_constituents))
perm_x = all_constituents[:n_perm_samples].float().to(device)
perm_mask = all_masks[:n_perm_samples].to(device)
perm_labels = labels[:n_perm_samples]
perm_weights = qcd_w[:n_perm_samples]

with torch.no_grad():
    out = model(perm_x, particle_mask=perm_mask)
    logits = out["classification"] if isinstance(out, dict) else out
    baseline_probs = torch.sigmoid(logits).squeeze().cpu().numpy()

if len(np.unique(perm_labels)) < 2:
    baseline_auc = np.nan
else:
    baseline_auc = roc_auc_score(perm_labels, baseline_probs, sample_weight=perm_weights)
print(f"  Baseline AUC: {baseline_auc:.4f}")

perm_importance = np.zeros(max_features_for_importance)
n_permutations = 3

for feat_idx in range(max_features_for_importance):
    auc_drops = []
    for _ in range(n_permutations):
        perm_x_copy = perm_x.clone()
        perm_indices = torch.randperm(n_perm_samples, device=perm_x.device)
        perm_x_copy[:, :, feat_idx] = perm_x[perm_indices, :, feat_idx]

        with torch.no_grad():
            out = model(perm_x_copy, particle_mask=perm_mask)
            logits = out["classification"] if isinstance(out, dict) else out
            perm_probs = torch.sigmoid(logits).squeeze().cpu().numpy()

        if len(np.unique(perm_labels)) < 2:
            auc_drops.append(0.0)
        else:
            try:
                perm_auc = roc_auc_score(perm_labels, perm_probs, sample_weight=perm_weights)
                auc_drops.append(float(baseline_auc - perm_auc))
            except Exception:
                auc_drops.append(0.0)

    perm_importance[feat_idx] = float(np.mean(auc_drops)) if len(auc_drops) > 0 else 0.0

# 3) Zero-ablation AUC-drop importance
print("3. Computing zero-ablation importance...")
ablation_importance = np.zeros(max_features_for_importance)
for feat_idx in range(max_features_for_importance):
    ablated_x = perm_x.clone()
    ablated_x[:, :, feat_idx] = 0.0
    with torch.no_grad():
        out = model(ablated_x, particle_mask=perm_mask)
        logits = out["classification"] if isinstance(out, dict) else out
        ablated_probs = torch.sigmoid(logits).squeeze().cpu().numpy()
    if len(np.unique(perm_labels)) < 2:
        ablation_importance[feat_idx] = 0.0
    else:
        try:
            ablation_auc = roc_auc_score(perm_labels, ablated_probs, sample_weight=perm_weights)
            ablation_importance[feat_idx] = float(baseline_auc - ablation_auc)
        except Exception:
            ablation_importance[feat_idx] = 0.0

# 4) Statistical separability
print("4. Computing statistical feature separability...")
stat_x = all_constituents[:, :, :max_features_for_importance].numpy()
stat_mask = all_masks.numpy().astype(bool)
stat_labels = labels

fisher_scores = np.zeros(max_features_for_importance)
ks_scores = np.zeros(max_features_for_importance)
for feat_idx in range(max_features_for_importance):
    sig_feat = stat_x[stat_labels == 1, :, feat_idx]
    sig_mask = stat_mask[stat_labels == 1]
    sig_vals = sig_feat[sig_mask].ravel()

    bkg_feat = stat_x[stat_labels == 0, :, feat_idx]
    bkg_mask_stat = stat_mask[stat_labels == 0]
    bkg_vals = bkg_feat[bkg_mask_stat].ravel()

    if len(sig_vals) == 0 or len(bkg_vals) == 0:
        fisher_scores[feat_idx] = 0.0
        ks_scores[feat_idx] = 0.0
        continue

    sig_mean, bkg_mean = float(sig_vals.mean()), float(bkg_vals.mean())
    sig_var, bkg_var = float(sig_vals.var() + 1e-8), float(bkg_vals.var() + 1e-8)
    fisher_scores[feat_idx] = (sig_mean - bkg_mean) ** 2 / (sig_var + bkg_var)

    n_subsample = min(10000, len(sig_vals), len(bkg_vals))
    sig_sample = np.random.choice(sig_vals, n_subsample, replace=False)
    bkg_sample = np.random.choice(bkg_vals, n_subsample, replace=False)
    ks_stat, _ = ks_2samp(sig_sample, bkg_sample)
    ks_scores[feat_idx] = float(ks_stat)


def normalize_scores(arr):
    arr = np.asarray(arr, dtype=float)
    finite = np.isfinite(arr)
    out = np.zeros_like(arr)
    if finite.sum() == 0:
        return out
    amin = arr[finite].min()
    amax = arr[finite].max()
    if amax - amin < 1e-12:
        return out
    out[finite] = (arr[finite] - amin) / (amax - amin)
    return out

combined_score = (
    normalize_scores(grad_importance)
    + normalize_scores(np.maximum(perm_importance, 0.0))
    + normalize_scores(np.maximum(ablation_importance, 0.0))
    + normalize_scores(fisher_scores)
    + normalize_scores(ks_scores)
) / 5.0

print("\nGenerating feature-importance plots...")
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
y = np.arange(max_features_for_importance)

sorted_idx = np.argsort(grad_importance)[::-1]
axes[0, 0].barh(y, grad_importance[sorted_idx], color="tab:blue", alpha=0.85)
axes[0, 0].set_yticks(y)
axes[0, 0].set_yticklabels([feature_names_used[i] for i in sorted_idx])
axes[0, 0].set_title("Gradient importance")
axes[0, 0].set_xlabel("Normalized mean |gradient|")
axes[0, 0].invert_yaxis()

sorted_idx = np.argsort(perm_importance)[::-1]
axes[0, 1].barh(y, perm_importance[sorted_idx], color="tab:green", alpha=0.85)
axes[0, 1].set_yticks(y)
axes[0, 1].set_yticklabels([feature_names_used[i] for i in sorted_idx])
axes[0, 1].set_title("Permutation AUC-drop importance")
axes[0, 1].set_xlabel("AUC drop when permuted")
axes[0, 1].axvline(0.0, color="black", linestyle="--", alpha=0.5)
axes[0, 1].invert_yaxis()

sorted_idx = np.argsort(ablation_importance)[::-1]
axes[1, 0].barh(y, ablation_importance[sorted_idx], color="tab:orange", alpha=0.85)
axes[1, 0].set_yticks(y)
axes[1, 0].set_yticklabels([feature_names_used[i] for i in sorted_idx])
axes[1, 0].set_title("Zero-ablation AUC-drop importance")
axes[1, 0].set_xlabel("AUC drop when zeroed")
axes[1, 0].axvline(0.0, color="black", linestyle="--", alpha=0.5)
axes[1, 0].invert_yaxis()

sorted_idx = np.argsort(combined_score)[::-1]
axes[1, 1].barh(y, combined_score[sorted_idx], color="tab:red", alpha=0.85)
axes[1, 1].set_yticks(y)
axes[1, 1].set_yticklabels([feature_names_used[i] for i in sorted_idx])
axes[1, 1].set_title("Combined feature ranking")
axes[1, 1].set_xlabel("Combined normalized score")
axes[1, 1].invert_yaxis()

plt.tight_layout()
save_fig(fig, "event_feature_importance_overview")
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(max_features_for_importance)
width = 0.42
ax.bar(x - width / 2, grad_importance_signal, width=width, label="Signal", color="tab:blue", alpha=0.8)
ax.bar(x + width / 2, grad_importance_background, width=width, label="Background", color="tab:orange", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(feature_names_used, rotation=45, ha="right")
ax.set_ylabel("Normalized mean |gradient|")
ax.set_title("Gradient importance split by class")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_fig(fig, "event_gradient_importance_by_class")
plt.show()

print("\nTop 5 features by combined score:")
top_n = min(5, len(feature_names_used))
for rank, idx in enumerate(np.argsort(combined_score)[::-1][:top_n], 1):
    print(f"  {rank}. {feature_names_used[idx]}: {combined_score[idx]:.4f}")

In [17]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss

print("=" * 70)
print("MODEL BEHAVIOR ANALYSIS")
print("=" * 70)

all_labels_eval = np.asarray(labels).astype(int).ravel()
all_outputs_eval = np.asarray(all_outputs).astype(float).ravel()

try:
    auc_global = roc_auc_score(all_labels_eval, all_outputs_eval)
except Exception:
    auc_global = np.nan

# 1) Error analysis at a default operating point.
print("\n1. ERROR ANALYSIS")
print("-" * 50)
threshold = 0.5
predictions = (all_outputs_eval > threshold).astype(int)

false_positives = (predictions == 1) & (all_labels_eval == 0)
false_negatives = (predictions == 0) & (all_labels_eval == 1)
true_positives = (predictions == 1) & (all_labels_eval == 1)
true_negatives = (predictions == 0) & (all_labels_eval == 0)

n_sig = int((all_labels_eval == 1).sum())
n_bkg = int((all_labels_eval == 0).sum())

sig_eff = 100.0 * true_positives.sum() / max(n_sig, 1)
bkg_rej = 100.0 * true_negatives.sum() / max(n_bkg, 1)

print(f"Threshold: {threshold:.2f}")
print(f"  True Positives:  {int(true_positives.sum()):>6} ({sig_eff:.1f}% of signal)")
print(f"  True Negatives:  {int(true_negatives.sum()):>6} ({bkg_rej:.1f}% of background)")
print(f"  False Positives: {int(false_positives.sum()):>6}")
print(f"  False Negatives: {int(false_negatives.sum()):>6}")

if false_positives.any() and false_negatives.any():
    print("\n  Misclassified event HT summary:")
    print(f"    False Positives mean HT: {ht[false_positives].mean():.1f} GeV")
    print(f"    False Negatives mean HT: {ht[false_negatives].mean():.1f} GeV")

# 2) Calibration diagnostics.
print("\n2. CALIBRATION")
print("-" * 50)
brier = brier_score_loss(all_labels_eval, all_outputs_eval)
logloss = log_loss(all_labels_eval, np.clip(all_outputs_eval, 1e-7, 1 - 1e-7))

prob_true, prob_pred = calibration_curve(
    all_labels_eval, all_outputs_eval, n_bins=10, strategy="uniform"
)

bin_edges = np.linspace(0.0, 1.0, 11)
ece = 0.0
for i in range(10):
    mask_bin = (all_outputs_eval >= bin_edges[i]) & (all_outputs_eval < bin_edges[i + 1])
    if mask_bin.sum() > 0:
        bin_acc = all_labels_eval[mask_bin].mean()
        bin_conf = all_outputs_eval[mask_bin].mean()
        ece += mask_bin.sum() * abs(bin_acc - bin_conf)
ece /= len(all_labels_eval)

print(f"  Brier score: {brier:.4f}")
print(f"  Log loss: {logloss:.4f}")
print(f"  ECE: {ece:.4f}")

# 3) Hard/easy sample analysis.
print("\n3. HARD SAMPLE ANALYSIS")
print("-" * 50)
uncertainty = np.abs(all_outputs_eval - 0.5)
hard_samples = uncertainty < 0.2
easy_samples = uncertainty >= 0.4

hard_frac = 100.0 * hard_samples.mean()
easy_frac = 100.0 * easy_samples.mean()
print(f"  Hard samples (score in [0.3, 0.7]): {hard_samples.sum()} ({hard_frac:.1f}%)")
print(f"  Easy samples (score < 0.1 or > 0.9): {easy_samples.sum()} ({easy_frac:.1f}%)")

if hard_samples.any():
    hard_acc = (predictions[hard_samples] == all_labels_eval[hard_samples]).mean()
    print(f"  Accuracy on hard samples: {100.0 * hard_acc:.1f}%")
if easy_samples.any():
    easy_acc = (predictions[easy_samples] == all_labels_eval[easy_samples]).mean()
    print(f"  Accuracy on easy samples: {100.0 * easy_acc:.1f}%")

n_constituents = all_masks.sum(dim=1).numpy()
if hard_samples.any() and easy_samples.any():
    print(f"  Mean constituents (hard/easy): {n_constituents[hard_samples].mean():.1f} / {n_constituents[easy_samples].mean():.1f}")
    print(f"  Mean HT (hard/easy): {ht[hard_samples].mean():.1f} / {ht[easy_samples].mean():.1f} GeV")

# 4) AUC vs HT slices.
print("\n4. AUC VS HT")
print("-" * 50)
quantile_edges = np.linspace(0.0, 1.0, 9)
ht_bins = np.quantile(ht, quantile_edges)
ht_bins = np.unique(ht_bins)
if len(ht_bins) < 2:
    ht_bins = np.array([ht.min(), ht.max() + 1e-6])

ht_auc = []
ht_bin_labels = []
for i in range(len(ht_bins) - 1):
    left = ht_bins[i]
    right = ht_bins[i + 1]
    mask = (ht >= left) & (ht < right)
    if mask.sum() < 50 or len(np.unique(all_labels_eval[mask])) < 2:
        ht_auc.append(np.nan)
    else:
        ht_auc.append(roc_auc_score(all_labels_eval[mask], all_outputs_eval[mask]))
    ht_bin_labels.append(f"{left:.0f}-{right:.0f}")

for bin_label, auc_val_bin in zip(ht_bin_labels, ht_auc):
    if np.isfinite(auc_val_bin):
        print(f"  HT {bin_label} GeV: AUC={auc_val_bin:.4f}")
    else:
        print(f"  HT {bin_label} GeV: AUC=nan (insufficient events)")

print("\nGenerating diagnostic figure...")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

ax = axes[0, 0]
ax.plot([0, 1], [0, 1], "k--", alpha=0.6, label="Perfect calibration")
ax.plot(prob_pred, prob_true, "o-", color="tab:blue", label="Model")
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Observed signal fraction")
ax.set_title("Calibration curve")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.hist(all_outputs_eval[all_labels_eval == 1], bins=50, range=(0, 1), histtype="step", density=True, color="tab:blue", label="Signal")
ax.hist(all_outputs_eval[all_labels_eval == 0], bins=50, range=(0, 1), histtype="step", density=True, color="tab:orange", label="Background")
ax.axvline(threshold, color="black", linestyle="--", alpha=0.6)
ax.axvspan(0.3, 0.7, color="gray", alpha=0.15, label="Hard region")
ax.set_xlabel("Classifier score")
ax.set_ylabel("Density")
ax.set_title("Output distribution")
ax.legend()

ax = axes[1, 0]
ax.bar(np.arange(len(ht_auc)), ht_auc, color="tab:green", alpha=0.85)
ax.set_xticks(np.arange(len(ht_auc)))
ax.set_xticklabels(ht_bin_labels, rotation=45, ha="right")
ax.set_ylim(0.45, 1.0)
ax.set_xlabel("HT bin [GeV]")
ax.set_ylabel("AUC")
ax.set_title("AUC across HT bins")
ax.grid(axis="y", alpha=0.3)

ax = axes[1, 1]
ax.axis("off")
summary_text = (
    f"Model behavior summary\n"
    f"{'=' * 30}\n"
    f"AUC (global): {auc_global:.4f}\n"
    f"Brier score:  {brier:.4f}\n"
    f"Log loss:     {logloss:.4f}\n"
    f"ECE:          {ece:.4f}\n\n"
    f"TP / TN: {int(true_positives.sum())} / {int(true_negatives.sum())}\n"
    f"FP / FN: {int(false_positives.sum())} / {int(false_negatives.sum())}\n\n"
    f"Hard samples: {hard_frac:.1f}%\n"
    f"Easy samples: {easy_frac:.1f}%\n"
)
ax.text(
    0.02,
    0.98,
    summary_text,
    va="top",
    ha="left",
    fontsize=11,
    family="monospace",
    bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.4},
)

plt.tight_layout()
save_fig(fig, "event_model_behavior_diagnostics")
plt.show()

In [32]:
import json
import importlib
import numpy as np
import matplotlib.pyplot as plt
import awkward as ak

from evaluation.roc import get_working_points
import evaluation.dihiggs as _dih

# Ensure awkward/vector Momentum4D arithmetic is available in this kernel session.
try:
    import vector
    ak.behavior.update(vector.backends.awkward.behavior)
except Exception:
    pass

# Reload to pick up latest helper changes in interactive notebook sessions.
importlib.reload(_dih)
from evaluation.dihiggs import (
    reconstruct_dihiggs_from_constituents,
    compute_significance_at_luminosity,
)
from evaluation.luminosity import load_physics_config, signal_weight, scale_qcd_weights_raw, scale_qcd_weights_per_event
import plotting.dihiggs_plots
importlib.reload(plotting.dihiggs_plots)
from plotting.dihiggs_plots import plot_mass_1d, plot_mass_2d

print("=" * 88)
print("EVENT-LEVEL DI-HIGGS MASS RECONSTRUCTION")
print("=" * 88)

# ----------------------------------------------------------------------------
# 1) Build loose/medium/tight from ROC, and define an ROC-based optimal point
# ----------------------------------------------------------------------------
roc_data = (fpr, tpr, auc_val, thresholds)
part_wps = get_working_points("Event ParT", roc_data)

physics = load_physics_config("hh-bbbb-obj-config.json")
LUMINOSITY_FB = physics["luminosity_fb"]
SIGNAL_XSEC_PB = physics["signal_xsec_pb"]
N_GEN_SIGNAL = physics["n_gen_signal"]

with open("hh-bbbb-obj-config.json", "r") as f:
    full_cfg = json.load(f)
sigma_to_ngen = {
    float(v["weight"]): int(v["n_gen"])
    for v in full_cfg["QCD_background"].values()
}

labels_np = np.asarray(labels).astype(int).ravel()
scores_np = np.asarray(scores).astype(float).ravel()
qcd_w_np = np.asarray(qcd_w).astype(float).ravel()

sig_all = labels_np == 1
bkg_all = labels_np == 0

sig_total_expected = float(
    np.sum(
        signal_weight(
            int(sig_all.sum()),
            luminosity_fb=LUMINOSITY_FB,
            signal_xsec_pb=SIGNAL_XSEC_PB,
            n_gen_signal=N_GEN_SIGNAL,
        )
    )
)
bkg_total_expected = float(
    np.sum(scale_qcd_weights_per_event(qcd_w_np[bkg_all], LUMINOSITY_FB))
)

sig_est_roc = (tpr * sig_total_expected) / np.sqrt(
    tpr * sig_total_expected + fpr * bkg_total_expected + 1e-12
)
opt_idx = int(np.nanargmax(sig_est_roc))
optimal_threshold = float(thresholds[opt_idx])

wp_thresholds = {
    "tight": float(part_wps[0]),
    "medium": float(part_wps[1]),
    "loose": float(part_wps[2]),
    "optimal": optimal_threshold,
}

print("\nROC-derived working points:")
print(
    f"{'WP':<10} {'threshold':>11} {'TPR':>10} {'FPR':>12} {'S/sqrt(S+B) est.':>20}"
)
print("-" * 70)
for wp_name, wp_thr in wp_thresholds.items():
    idx_wp = int(np.argmin(np.abs(thresholds - wp_thr)))
    print(
        f"{wp_name:<10} {wp_thr:>11.5f} {tpr[idx_wp]:>10.4f} {fpr[idx_wp]:>12.4e} {sig_est_roc[idx_wp]:>20.4f}"
    )

# Choose between: "loose" | "medium" | "tight" | "optimal"
WP_SELECTION = "loose"
EVENT_TAG_THRESHOLD = wp_thresholds[WP_SELECTION]
print(f"\nSelected working point: {WP_SELECTION} (event score >= {EVENT_TAG_THRESHOLD:.5f})")

# ----------------------------------------------------------------------------
# 2) Select events by event-level hh-tag, then cluster constituents to jets
# ----------------------------------------------------------------------------
selected_events = scores_np >= EVENT_TAG_THRESHOLD
n_selected = int(selected_events.sum())
print(f"Selected events at WP: {n_selected:,} / {len(scores_np):,}")

selected_constits = all_constituents[selected_events].cpu().numpy()
selected_masks = all_masks[selected_events].cpu().numpy().astype(bool)
selected_labels = labels_np[selected_events]
selected_qcd_raw = qcd_w_np[selected_events]

reco = reconstruct_dihiggs_from_constituents(
    selected_constits,
    masks=selected_masks,
    top_k=4,
    jet_R=0.4,
    min_jet_pt=25.0,
)

reco_event_idx = reco["event_indices"]
if len(reco_event_idx) == 0:
    raise RuntimeError(
        "No selected events had >=4 clustered jets. Try a looser WP or lower min_jet_pt."
    )

reco_labels = selected_labels[reco_event_idx]
reco_qcd_raw = selected_qcd_raw[reco_event_idx]

sig_mask_reco = reco_labels == 1
bkg_mask_reco = reco_labels == 0

sig_lead_m = reco["lead_m"][sig_mask_reco]
sig_sub_m = reco["sub_m"][sig_mask_reco]
sig_hh_m = reco["hh_m"][sig_mask_reco]

bkg_lead_m = reco["lead_m"][bkg_mask_reco]
bkg_sub_m = reco["sub_m"][bkg_mask_reco]
bkg_hh_m = reco["hh_m"][bkg_mask_reco]
bkg_weights_raw = reco_qcd_raw[bkg_mask_reco]

print("\nReconstruction summary:")
print(f"  Events with >=4 clustered jets: {len(reco_event_idx):,}")
print(f"  Signal events reconstructed:    {len(sig_lead_m):,}")
print(f"  Background events reconstructed:{len(bkg_lead_m):,}")

# ----------------------------------------------------------------------------
# 3) Plot di-Higgs mass distributions in test_trained_part style
# ----------------------------------------------------------------------------
plot_label = f"Event ParT ({WP_SELECTION}={EVENT_TAG_THRESHOLD:.4f})"

sig_weights_expected = signal_weight(
    len(sig_lead_m),
    luminosity_fb=LUMINOSITY_FB,
    signal_xsec_pb=SIGNAL_XSEC_PB,
    n_gen_signal=N_GEN_SIGNAL,
)
bkg_weights_expected = scale_qcd_weights_per_event(
    bkg_weights_raw,
    # sigma_to_ngen,
    luminosity_fb=LUMINOSITY_FB,
)

fig1 = plot_mass_1d(
    sig_lead_mass=sig_lead_m,
    sig_sub_mass=sig_sub_m,
    sig_hh_mass=sig_hh_m,
    qcd_lead_mass=bkg_lead_m,
    qcd_sub_mass=bkg_sub_m,
    qcd_hh_mass=bkg_hh_m,
    sig_weights=sig_weights_expected,
    qcd_weights=bkg_weights_expected,
    label=plot_label,
    sig_window_h=(90, 160),
    sig_window_hh=(250, 550),
    density=True,
)
save_fig(fig1, f"event_dihiggs_mass_1d_{WP_SELECTION}")
plt.show()

fig2 = plot_mass_2d(
    sig_lead_mass=sig_lead_m,
    sig_sub_mass=sig_sub_m,
    qcd_lead_mass=bkg_lead_m,
    qcd_sub_mass=bkg_sub_m,
    sig_weights=sig_weights_expected,
    qcd_weights=bkg_weights_expected,
    label=plot_label,
    r_hh_cut=55.0,
    mh_centers=(125.0, 120.0),
)
save_fig(fig2, f"event_dihiggs_mass_2d_{WP_SELECTION}")
plt.show()

# ----------------------------------------------------------------------------
# 4) Significance (rectangular mHH and circular R_HH regions)
# ----------------------------------------------------------------------------
if len(sig_lead_m) > 0 and len(bkg_lead_m) > 0:
    sig_rect = compute_significance_at_luminosity(
        sig_lead_m,
        sig_sub_m,
        bkg_lead_m,
        bkg_sub_m,
        bkg_raw_weights=bkg_weights_raw,
        sigma_to_ngen=sigma_to_ngen,
        n_gen_signal=N_GEN_SIGNAL,
        luminosity_fb=LUMINOSITY_FB,
        signal_xsec_pb=SIGNAL_XSEC_PB,
        region="rectangular",
        rect_window=(250, 550),
        convention="b"
    )

    sig_circ = compute_significance_at_luminosity(
        sig_lead_m,
        sig_sub_m,
        bkg_lead_m,
        bkg_sub_m,
        bkg_raw_weights=bkg_weights_raw,
        sigma_to_ngen=sigma_to_ngen,
        n_gen_signal=N_GEN_SIGNAL,
        luminosity_fb=LUMINOSITY_FB,
        signal_xsec_pb=SIGNAL_XSEC_PB,
        region="circular",
        r_hh_cut=55.0,
        convention="b"
    )

    print("\nSignificance summary:")
    print(
        f"  Rectangular (250 < m_HH < 550): S={sig_rect['S']:.2f}, B={sig_rect['B']:.2e}, "
        f"S/sqrt(S+B)={sig_rect['significance']:.4f}"
    )
    print(
        f"  Circular (R_HH < 55):           S={sig_circ['S']:.2f}, B={sig_circ['B']:.2e}, "
        f"S/sqrt(S+B)={sig_circ['significance']:.4f}"
    )
else:
    print("\nInsufficient reconstructed signal/background events for significance.")

print(f"\nSaved plots in: {plot_dir}")